In [1]:
import requests
import pandas as pd
import numpy as np 
import glob

In [ ]:
city_df1= pd.read_csv("C:\\Users\\Shree\\OneDrive\\Documents\\NYC Citi Bike Trip Data_Project\\data\\202608-citibike-tripdata_1.csv")
city_df2 = pd.read_csv("C:\\Users\\Shree\\OneDrive\\Documents\\NYC Citi Bike Trip Data_Project\\data\\202608-citibike-tripdata_2.csv")
city_df3 = pd.read_csv("C:\\Users\\Shree\\OneDrive\\Documents\\NYC Citi Bike Trip Data_Project\\data\\202608-citibike-tripdata_3.csv")

weather_df = pd.read_csv("C:\\Users\\Shree\\OneDrive\\Documents\\NYC Citi Bike Trip Data_Project\\data\\nyc_weather.csv")
weather_df

station_df = pd.read_csv("C:\\Users\\Shree\\OneDrive\\Documents\\NYC Citi Bike Trip Data_Project\\data\\station_information.csv")
station_df

In [ ]:
# Combining three files into one 

comb_citi=pd.concat([city_df1,city_df2,city_df3],ignore_index=True)
comb_citi

In [ ]:
print("Trips shape:", comb_citi.shape)
print("Weather shape:", weather_df.shape)
print("Stations shape:", station_df.shape)

In [ ]:
# STEP 2: Clean Trip Data
# Python

# Convert to datetime

comb_citi['started_at'] = pd.to_datetime(comb_citi['started_at'])

In [ ]:
comb_citi['ended_at'] = pd.to_datetime(comb_citi['ended_at'])

In [ ]:
# Calculate trip duration in minutes
comb_citi['trip_duration_min'] = (comb_citi['ended_at'] - comb_citi['started_at']).dt.total_seconds() / 60

# Remove invalid trips
comb_citi = comb_citi[(comb_citi['trip_duration_min'] > 1) & (comb_citi['trip_duration_min'] < 24*60)]

# Remove duplicates
comb_citi = comb_citi.drop_duplicates(subset=['ride_id'])

print("After cleaning trips:", comb_citi.shape)

In [ ]:
# STEP 3: Clean Station Data
# Python
# Keep only useful columns

station_df = station_df[['station_id', 'name', 'lat', 'lon', 'capacity']].copy()

# Rename columns
station_df = station_df.rename(columns={
'name': 'station_name',
'lat': 'latitude',
'lon': 'longitude'
})

In [ ]:
# STEP 4: Merge Trips + Station Information
# Python
# ----- Start Station Merge -----


comb_citi = comb_citi.merge(
station_df,
left_on='start_station_id',
right_on='station_id',
how='left'
)

comb_citi = comb_citi.rename(columns={
'station_name': 'start_station_name_clean',
'latitude': 'start_lat_station',
'longitude': 'start_lng_station',
'capacity': 'start_capacity'
})
comb_citi = comb_citi.drop(columns=['station_id'])

In [ ]:
# ----- End Station Merge -----

comb_citi = comb_citi.merge(
station_df,
left_on='end_station_id',
right_on='station_id',
how='left'
)

comb_citi = comb_citi.rename(columns={
'station_name': 'end_station_name_clean',
'latitude': 'end_lat_station',
'longitude': 'end_lng_station',
'capacity': 'end_capacity'

})
comb_citi = comb_citi.drop(columns=['station_id'])


In [ ]:
# STEP 5: Merge with Weather Data
# Python
# Prepare weather

weather_df['time'] = pd.to_datetime(weather_df['time'])
weather_df['date_hour'] = weather_df['time'].dt.floor('h')

# Prepare trips
comb_citi['date_hour'] = comb_citi['started_at'].dt.floor('h')

# Merge
final_df = comb_citi.merge(weather_df, on='date_hour', how='left')

In [ ]:
# Ensure datetime conversion

final_df['started_at'] = pd.to_datetime(final_df['started_at'])
final_df['ended_at'] = pd.to_datetime(final_df['ended_at'])

# Trip duration (minutes)

final_df['trip_duration_min'] = (final_df['ended_at'] - final_df['started_at']).dt.total_seconds() / 60

# Trip distance (Haversine formula)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

final_df['trip_distance_km'] = final_df.apply(
    lambda row: haversine(row['start_lat'], row['start_lng'], row['end_lat'], row['end_lng']),
    axis=1
)

# Time features

final_df['hour'] = final_df['started_at'].dt.hour
final_df['day_of_week'] = final_df['started_at'].dt.day_name()
final_df['is_weekend'] = final_df['started_at'].dt.day_of_week >= 5

# Capacity placeholders (if not available in dataset)

final_df['start_capacity'] = np.nan
final_df['end_capacity'] = np.nan

# Final column selection

final_columns = [
    'ride_id', 'rideable_type', 'started_at', 'ended_at',
    'trip_duration_min', 'trip_distance_km', 'member_casual',

    'start_station_id', 'start_station_name', 'start_lat', 'start_lng', 'start_capacity',
    'end_station_id', 'end_station_name', 'end_lat', 'end_lng', 'end_capacity',

    'hour', 'day_of_week', 'is_weekend',

    'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m'
]

final_df = final_df[final_columns]

print("Final shape:", final_df.shape)
print(final_df.head())


In [ ]:
# Save Cleaned data

final_df.to_csv("Combined_Data_Cleaned_Dataset2.csv")